# Task 1 – Data Cleaning and Preprocessing
**Internship:** Auspify Technologies – Data Science Program  
**Intern:** Shabbir Moomin (Colombo, Sri Lanka)  
**Dataset:** Netflix titles (`Dataset.csv`)

**Workflow**
1. Import the dataset using Pandas
2. Identify missing and duplicate records
3. Handle null values and inconsistent formats
4. Transform categorical and date-related columns
5. Create a clean dataset for further analysis (used in Task 2 – EDA)

## Step 1 – Import the dataset using Pandas

In [1]:
# Python Language - Initial command steps (First Version)
import pandas as pd
import numpy as np

# show every column when printing, otherwise pandas hides the middle ones (annoying!)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 150)

df = pd.read_csv('Dataset.csv')
raw = df.copy()          # keep an untouched copy so we can compare before/after at the end
print('Shape:', df.shape)
df.head()

Shape: (8790, 10)


,show_id,type,title,director,country,date_added,release_year,rating,duration,listed_in
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,United States,9/25/2021,2020,PG-13,90 min,Documentaries
1,s3,TV Show,Ganglands,Julien Leclercq,France,9/24/2021,2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act..."
2,s6,TV Show,Midnight Mass,Mike Flanagan,United States,9/24/2021,2021,TV-MA,1 Season,"TV Dramas, TV Horror, TV Mysteries"
3,s14,Movie,Confessions of an Invisible Girl,Bruno Garotti,Brazil,9/22/2021,2021,TV-PG,91 min,"Children & Family Movies, Comedies"
4,s8,Movie,Sankofa,Haile Gerima,United States,9/24/2021,1993,TV-MA,125 min,"Dramas, Independent Movies, International Movies"


In [2]:
# quick look at structure: dtypes, non-null counts, memory
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8790 entries, 0 to 8789
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   show_id       8790 non-null   object
 1   type          8790 non-null   object
 2   title         8790 non-null   object
 3   director      8790 non-null   object
 4   country       8790 non-null   object
 5   date_added    8790 non-null   object
 6   release_year  8790 non-null   int64 
 7   rating        8790 non-null   object
 8   duration      8790 non-null   object
 9   listed_in     8790 non-null   object
dtypes: int64(1), object(9)
memory usage: 686.8+ KB


In [3]:
# summary stats (numeric) and (categorical) - first impressions
display(df.describe())
display(df.describe(include='object'))

,release_year
count,8790.000000
mean,2014.183163
std,8.825466
min,1925.000000
25%,2013.000000
50%,2017.000000
75%,2019.000000
max,2021.000000


,show_id,type,title,director,country,date_added,rating,duration,listed_in
count,8790,8790,8790,8790,8790,8790,8790,8790,8790
unique,8790,2,8787,4528,86,1713,14,220,513
top,s1,Movie,9-Feb,Not Given,United States,1/1/2020,TV-MA,1 Season,"Dramas, International Movies"
freq,1,6126,2,2588,3240,110,3205,1791,362


## Step 2 – Identify missing and duplicate records
At first glance `isnull()` shows zero nulls, but that can be misleading. Datasets like this often hide
missing values behind **placeholder text**, so we check for those too.

In [4]:
# 2a) Standard null check
print('Null values per column:')
print(df.isnull().sum())

Null values per column:
show_id         0
type            0
title           0
director        0
country         0
date_added      0
release_year    0
rating          0
duration        0
listed_in       0
dtype: int64


In [5]:
# 2b) Hidden missing values: placeholders like 'Not Given', 'Unknown', 'N/A', empty strings
placeholders = ['not given', 'unknown', 'n/a', 'na', 'none', 'null', '-', '']

hidden = {}
for col in df.select_dtypes(include='object').columns:
    mask = df[col].astype(str).str.strip().str.lower().isin(placeholders)
    if mask.sum() > 0:
        hidden[col] = int(mask.sum())

print('Placeholder-style missing values:', hidden)
print({k: f'{v/len(df):.1%}' for k, v in hidden.items()})

Placeholder-style missing values: {'title': 1, 'director': 2588, 'country': 287}
{'title': '0.0%', 'director': '29.4%', 'country': '3.3%'}


In [6]:
# 2c) Duplicate records
print('Exact duplicate rows            :', df.duplicated().sum())
print('Duplicate show_id               :', df['show_id'].duplicated().sum())

# show_id is unique, so a plain duplicated() can miss "same title, different ID" rows.
# Compare every column EXCEPT show_id:
content_cols = [c for c in df.columns if c != 'show_id']
dup_mask = df.duplicated(subset=content_cols, keep=False)
print('Duplicates ignoring show_id     :', df.duplicated(subset=content_cols).sum())
df[dup_mask].sort_values('title')

Exact duplicate rows            : 0
Duplicate show_id               : 0
Duplicates ignoring show_id     : 3


,show_id,type,title,director,country,date_added,release_year,rating,duration,listed_in
2925,s3963,Movie,15-Aug,Swapnaneel Jayakar,India,3/29/2019,2019,TV-14,124 min,"Comedies, Dramas, Independent Movies"
4261,s5967,Movie,15-Aug,Swapnaneel Jayakar,India,3/29/2019,2019,TV-14,124 min,"Comedies, Dramas, Independent Movies"
3285,s4523,Movie,22-Jul,Paul Greengrass,Norway,10/10/2018,2018,R,144 min,"Dramas, Thrillers"
4260,s5966,Movie,22-Jul,Paul Greengrass,Norway,10/10/2018,2018,R,144 min,"Dramas, Thrillers"
393,s3997,TV Show,9-Feb,Not Given,Pakistan,3/20/2019,2018,TV-14,1 Season,"International TV Shows, TV Dramas"
537,s5965,TV Show,9-Feb,Not Given,Pakistan,3/20/2019,2018,TV-14,1 Season,"International TV Shows, TV Dramas"


In [7]:
# 2d) Hidden formatting problems: leading/trailing whitespace
for col in df.select_dtypes(include='object').columns:
    n = (df[col] != df[col].str.strip()).sum()
    if n:
        print(f'{col}: {n} value(s) with extra whitespace ->', df.loc[df[col] != df[col].str.strip(), col].tolist())

title: 1 value(s) with extra whitespace -> ['Consequences\xa0']


In [8]:
# 2e) Inconsistent formats - inspect categorical values & the date column
print(df['type'].value_counts(), '\n')
print(df['rating'].value_counts(), '\n')
print('date_added sample:', df['date_added'].head(3).tolist())
print('duration units   :', df['duration'].str.extract(r'([A-Za-z]+)$')[0].value_counts().to_dict())

Movie      6126
TV Show    2664
Name: type, dtype: int64 

TV-MA       3205
TV-14       2157
TV-PG        861
R            799
PG-13        490
TV-Y7        333
TV-Y         306
PG           287
TV-G         220
NR            79
G             41
TV-Y7-FV       6
NC-17          3
UR             3
Name: rating, dtype: int64 

date_added sample: ['9/25/2021', '9/24/2021', '9/24/2021']
duration units   : {'min': 6126, 'Season': 1791, 'Seasons': 873}


### Findings
- No true `NaN` values, **but** `director` (~29%) and `country` (~3%) use the placeholder `"Not Given"`.
- 3 duplicate records exist: same content, different `show_id` (e.g. *9-Feb*, *15-Aug*, *22-Jul*).
- 1 title has trailing whitespace (`"Consequences "`) - this one also hides a **4th duplicate**, which only shows up after the whitespace is stripped in Step 3 (that's why 4 rows get removed, not 3).
- `date_added` is stored as text (`m/d/YYYY`), `duration` mixes minutes and seasons in one column.
- `listed_in` and `country` hold multiple comma-separated values in a single cell.

## Step 3 – Handle null values and inconsistent formats

In [9]:
# 3a) strip whitespace from every text column (fixes the 'Consequences ' problem)
text_cols = df.select_dtypes(include='object').columns
for col in text_cols:
    df[col] = df[col].str.strip()

# 3b) convert placeholders to real NaN first, so pandas can treat them properly
df = df.replace({'Not Given': np.nan})
print(df.isnull().sum())

show_id            0
type               0
title              0
director        2588
country          287
date_added         0
release_year       0
rating             0
duration           0
listed_in          0
dtype: int64


In [10]:
# 3c) decide how to treat missing values
# director/country are categorical text -> deleting ~2.9k rows would throw away a lot of good data,
# and guessing a director/country would be inventing facts. So we label them 'Unknown'.
df['director'] = df['director'].fillna('Unknown')
df['country']  = df['country'].fillna('Unknown')

print('Remaining nulls:', df.isnull().sum().sum())

Remaining nulls: 0


In [11]:
# 3d) remove duplicates (keep the first show_id of each pair)
before = len(df)
df = df.drop_duplicates(subset=content_cols, keep='first').reset_index(drop=True)
print(f'Rows before: {before} | after: {len(df)} | removed: {before - len(df)}')

Rows before: 8790 | after: 8786 | removed: 4


In [12]:
# 3e) fix data types / formats
# date_added -> proper datetime. format given explicitly so pandas doesn't guess day vs month
df['date_added'] = pd.to_datetime(df['date_added'], format='%m/%d/%Y')

# categorical columns -> 'category' dtype (saves memory and is cleaner for analysis)
for col in ['type', 'rating']:
    df[col] = df[col].astype('category')

# NOTE: deliberately NOT changing title casing - brand names like 'iZombie' would get mangled
df.dtypes

show_id                 object
type                  category
title                   object
director                object
country                 object
date_added      datetime64[ns]
release_year             int64
rating                category
duration                object
listed_in               object
dtype: object

## Step 4 – Transform categorical and date-related columns

In [13]:
# ---------- 4a) Date features ----------
df['year_added']    = df['date_added'].dt.year
df['month_added']   = df['date_added'].dt.month
df['month_name']    = df['date_added'].dt.month_name()
df['day_of_week']   = df['date_added'].dt.day_name()
df['quarter_added'] = df['date_added'].dt.quarter

# how long after release did Netflix add it? (negative = added before official release year -> clip at 0)
df['years_to_add'] = (df['year_added'] - df['release_year']).clip(lower=0)

df[['date_added', 'year_added', 'month_name', 'day_of_week', 'years_to_add']].head()

,date_added,year_added,month_name,day_of_week,years_to_add
0,2021-09-25,2021,September,Saturday,1
1,2021-09-24,2021,September,Friday,0
2,2021-09-24,2021,September,Friday,0
3,2021-09-22,2021,September,Wednesday,0
4,2021-09-24,2021,September,Friday,28


In [14]:
# ---------- 4b) Duration: split into a number and a unit ----------
# '90 min' -> 90 / 'min'   |   '2 Seasons' -> 2 / 'Season'
dur = df['duration'].str.extract(r'(\d+)\s*(\w+)')
df['duration_value'] = dur[0].astype(int)

# separate numeric columns for movies vs. TV shows - mixing them in one column ruins averages
df['duration_min'] = np.where(df['type'] == 'Movie',   df['duration_value'], np.nan)
df['seasons']      = np.where(df['type'] == 'TV Show', df['duration_value'], np.nan)
df = df.drop(columns=['duration_value'])

# sanity check: every Movie should have minutes, every TV Show should have seasons
print(df.groupby('type', observed=True)[['duration_min', 'seasons']].agg(['count', 'mean']).round(1))

        duration_min       seasons     
               count  mean   count mean
type                                   
Movie           6123  99.6       0  NaN
TV Show            0   NaN    2663  1.8


In [15]:
# ---------- 4c) Multi-value columns: country & listed_in ----------
# first country listed = 'primary' country; also count how many countries co-produced it
df['primary_country'] = df['country'].str.split(',').str[0].str.strip()
df['n_countries']     = df['country'].apply(lambda x: 0 if x == 'Unknown' else len(x.split(',')))

# genres: keep a python list + the first genre as the main one
df['genre_list']    = df['listed_in'].str.split(',').apply(lambda g: [i.strip() for i in g])
df['primary_genre'] = df['genre_list'].str[0]
df['n_genres']      = df['genre_list'].str.len()

df[['country', 'primary_country', 'n_countries', 'listed_in', 'primary_genre', 'n_genres']].head()

,country,primary_country,n_countries,listed_in,primary_genre,n_genres
0,United States,United States,1,Documentaries,Documentaries,1
1,France,France,1,"Crime TV Shows, International TV Shows, TV Act...",Crime TV Shows,3
2,United States,United States,1,"TV Dramas, TV Horror, TV Mysteries",TV Dramas,3
3,Brazil,Brazil,1,"Children & Family Movies, Comedies",Children & Family Movies,2
4,United States,United States,1,"Dramas, Independent Movies, International Movies",Dramas,3


In [16]:
# ---------- 4d) Rating -> audience group ----------
# 14 raw rating codes is too granular for analysis, so bucket them (US TV/movie rating systems)
rating_map = {
    'TV-Y': 'Kids', 'TV-Y7': 'Kids', 'TV-Y7-FV': 'Kids', 'TV-G': 'Kids', 'G': 'Kids',
    'TV-PG': 'Older Kids', 'PG': 'Older Kids',
    'TV-14': 'Teens', 'PG-13': 'Teens',
    'TV-MA': 'Adults', 'R': 'Adults', 'NC-17': 'Adults',
    'NR': 'Not Rated', 'UR': 'Not Rated',
}
df['audience'] = df['rating'].astype(str).map(rating_map)
print('Unmapped ratings:', df['audience'].isnull().sum())   # should be 0
df['audience'].value_counts()

Unmapped ratings: 0


Adults        4005
Teens         2645
Older Kids    1148
Kids           906
Not Rated       82
Name: audience, dtype: int64

In [17]:
# ---------- 4e) Encode categorical variables ----------
# binary label for type (handy for ML later)
df['is_movie'] = (df['type'] == 'Movie').astype(int)

# ordered category for audience group
df['audience'] = pd.Categorical(df['audience'],
                                categories=['Kids', 'Older Kids', 'Teens', 'Adults', 'Not Rated'],
                                ordered=True)

# one-hot encoding for audience (kept separate so the main clean file stays readable)
audience_dummies = pd.get_dummies(df['audience'], prefix='aud', dtype=int)
print(audience_dummies.head())

# genre flags: one 0/1 column per top-10 genre (a title can have several genres)
top_genres = pd.Series([g for lst in df['genre_list'] for g in lst]).value_counts().head(10).index
for g in top_genres:
    df['genre_' + g.lower().replace(' ', '_').replace('&', 'and')] = df['genre_list'].apply(lambda x: int(g in x))
print('\nTop genres:', list(top_genres))

   aud_Kids  aud_Older Kids  aud_Teens  aud_Adults  aud_Not Rated
0         0               0          1           0              0
1         0               0          0           1              0
2         0               0          0           1              0
3         0               1          0           0              0
4         0               0          0           1              0

Top genres: ['International Movies', 'Dramas', 'Comedies', 'International TV Shows', 'Documentaries', 'Action & Adventure', 'TV Dramas', 'Independent Movies', 'Children & Family Movies', 'Romantic Movies']


## Step 5 – Create a clean dataset for further analysis

In [18]:
# final validation checks
print('Shape (raw -> clean):', raw.shape, '->', df.shape)
print('Nulls (excluding movie/TV-specific cols):',
      df.drop(columns=['duration_min', 'seasons']).isnull().sum().sum())
print('Duplicate rows:', df.duplicated(subset=content_cols).sum())
print('Unique show_id:', df['show_id'].is_unique)
print('Date range    :', df['date_added'].min().date(), 'to', df['date_added'].max().date())

Shape (raw -> clean): (8790, 10) -> (8786, 35)
Nulls (excluding movie/TV-specific cols): 0
Duplicate rows: 0
Unique show_id: True
Date range    : 2008-01-01 to 2021-09-25


In [19]:
# genre_list is a python list - CSV can't store it nicely, so drop it from the exported file
clean = df.drop(columns=['genre_list'])
clean.to_csv('Netflix_Cleaned.csv', index=False)

# quick reload test to make sure the file is fine
check = pd.read_csv('Netflix_Cleaned.csv', parse_dates=['date_added'])
print('Saved & reloaded OK ->', check.shape)
check.head()

Saved & reloaded OK -> (8786, 34)


,show_id,type,title,director,country,date_added,release_year,rating,duration,listed_in,year_added,month_added,month_name,day_of_week,quarter_added,years_to_add,duration_min,seasons,primary_country,n_countries,primary_genre,n_genres,audience,is_movie,genre_international_movies,genre_dramas,genre_comedies,genre_international_tv_shows,genre_documentaries,genre_action_and_adventure,genre_tv_dramas,genre_independent_movies,genre_children_and_family_movies,genre_romantic_movies
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,United States,2021-09-25,2020,PG-13,90 min,Documentaries,2021,9,September,Saturday,3,1,90.0,NaN,United States,1,Documentaries,1,Teens,1,0,0,0,0,1,0,0,0,0,0
1,s3,TV Show,Ganglands,Julien Leclercq,France,2021-09-24,2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",2021,9,September,Friday,3,0,NaN,1.0,France,1,Crime TV Shows,3,Adults,0,0,0,0,1,0,0,0,0,0,0
2,s6,TV Show,Midnight Mass,Mike Flanagan,United States,2021-09-24,2021,TV-MA,1 Season,"TV Dramas, TV Horror, TV Mysteries",2021,9,September,Friday,3,0,NaN,1.0,United States,1,TV Dramas,3,Adults,0,0,0,0,0,0,0,1,0,0,0
3,s14,Movie,Confessions of an Invisible Girl,Bruno Garotti,Brazil,2021-09-22,2021,TV-PG,91 min,"Children & Family Movies, Comedies",2021,9,September,Wednesday,3,0,91.0,NaN,Brazil,1,Children & Family Movies,2,Older Kids,1,0,0,1,0,0,0,0,0,1,0
4,s8,Movie,Sankofa,Haile Gerima,United States,2021-09-24,1993,TV-MA,125 min,"Dramas, Independent Movies, International Movies",2021,9,September,Friday,3,28,125.0,NaN,United States,1,Dramas,3,Adults,1,1,1,0,0,0,0,0,1,0,0


## Summary of cleaning performed
| Issue found | Action taken |
|---|---|
| `"Not Given"` placeholder in `director` (2,588) and `country` (287) | Converted to NaN, then labelled `Unknown` |
| 4 duplicate records with different `show_id` (3 obvious + 1 hidden by whitespace) | Removed (kept first occurrence) |
| Whitespace in text (`"Consequences "`) | Stripped across all text columns |
| `date_added` stored as text | Converted to `datetime`; added year / month / weekday / quarter |
| `duration` mixes minutes & seasons | Split into `duration_min` (movies) and `seasons` (TV shows) |
| Multi-value `country` / `listed_in` | Extracted `primary_country`, `primary_genre`, counts, and top-10 genre flags |
| 14 granular ratings | Grouped into `audience` (ordered category) |
| Text categories | `type` → `is_movie`; audience one-hot available |

The output **`Netflix_Cleaned.csv`** is ready for **Task 2 – Exploratory Data Analysis**.